# QA de Prospección Digital

Recorrido por los hallazgos de la prueba de campo. **La lógica vive en `src/`**;
este cuaderno la ejecuta y muestra las tablas, para que análisis y reporte no se
desincronicen.

Para regenerar todo sin abrir el notebook:

```bash
python run.py guatemala
```

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # el repo, no notebooks/

import pandas as pd

from src import config, figuras
from src.analisis import ejecutar, resumen_categoria

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 200)

PAIS = config.pais("guatemala")
PAIS

Pais(slug='guatemala', nombre='Guatemala', iso3='GTM', patron_clientes='Clientes_*.csv', patron_universo='Dataplor_*.csv', patron_respuestas='Respuestas*.xlsx', lat_centro=15.5, bbox=(-92.4, -88.1, 13.5, 18.0), divisiones='departamentos')

---
## 1. Ejecutar el análisis

`ejecutar()` corre las cuatro ramas del formulario, escribe los CSV en
`output/findings/` y devuelve un `Resultados` con las tablas intermedias
(`res["validado"]`, `res["ooc"]`, …) y las métricas (`res.metrics`).

In [2]:
res = ejecutar(PAIS)
f = res.fuentes
m = res.metrics

<Fuentes Guatemala: 104 respuestas, 100 POI unicos, 34,915 universo, 229,077 maestra>
  -> errores_id.csv  (8 filas)


  -> existe_como_cliente_validado.csv  (16 filas)
  -> pos_id_ausente_en_corte.csv  (2 filas)
  -> cruces_a_revisar.csv  (2 filas)


  -> titular_vs_nombre_comercial.csv  (14 filas)
  -> ejemplos_hallazgo3.csv  (7 filas)
  -> poi_inexistentes.csv  (35 filas)


  -> fuera_de_cobertura.csv  (16 filas)
  -> errores_segmentacion.csv  (16 filas)
  -> cuenta_de_las_visitas.csv  (7 filas)
  -> hallazgos_geo.csv  (100 filas)
  -> hallazgos_resumen.csv  (8 filas)
  -> Respuestas Guatemala - con causa.xlsx  (104 filas)


---
## 2. Integridad del formulario

Antes de leer resultados hay que auditar el insumo.

In [3]:
print(f"respuestas recibidas   : {m['respuestas_crudas']}")
print(f"POI únicos             : {m['visitas_unicas']}")
print(f"encuestados dos veces  : {m['ids_duplicados']}")
print(f"IDs ausentes del universo: {m['ids_huerfanos']}")
print(f"con coordenadas válidas: {m['coords_validas']}")

res["errores_id"][["dataplor_id", "Timestamp", f.C_NOMBRE, f.C_VISITA]]

respuestas recibidas   : 104
POI únicos             : 100
encuestados dos veces  : 4
IDs ausentes del universo: 0
con coordenadas válidas: 100


,dataplor_id,Timestamp,Código de Cliente interno y Nombre de Establecimiento,DATOS DE VISITA
32,2ddc6c5e-d8e5-4e3d-98cc-d0b0540c12ae,2026-08-24 19:38:51,936.Monjas,NO EXISTE EL ESTABLECIMIENTO
33,2ddc6c5e-d8e5-4e3d-98cc-d0b0540c12ae,2026-08-24 19:58:00,936.Monjas,NO EXISTE EL ESTABLECIMIENTO
37,5ff5c2ab-75f8-47ce-8c1d-45414c1c041d,2026-08-24 19:31:05,1953.Restaurante Doña Esther,NO EXISTE EL ESTABLECIMIENTO
39,5ff5c2ab-75f8-47ce-8c1d-45414c1c041d,2026-08-24 20:00:03,1953.Restaurante Doña Esther,NO EXISTE EL ESTABLECIMIENTO
81,7fa113fc-fdf2-427c-82e9-2942cacb438d,2026-08-25 08:13:45,450.Polideportivo Municipal de Asunción Mita,OTROS RUBROS
99,7fa113fc-fdf2-427c-82e9-2942cacb438d,2026-08-28 06:15:10,450.Polideportivo Municipal de Asunción Mita,OTROS RUBROS
42,c94adf8e-c26f-4973-8862-23cc32ff3e6b,2026-08-24 19:46:42,2647.Casa Campestre,EXISTE COMO CLIENTE
43,c94adf8e-c26f-4973-8862-23cc32ff3e6b,2026-08-24 20:01:12,2647.Casa Campestre,EXISTE COMO CLIENTE


---
## 3. Panorama general

In [4]:
display(res["distribucion"])
display(resumen_categoria(f.respuestas["macro_resultado"], "Macro resultado"))
print(f"tasa de apertura: {m['tasa_apertura']}%")

,Cantidad,Porcentaje (%)
DATOS DE VISITA,,
NO EXISTE EL ESTABLECIMIENTO,35,35.0
EXISTE COMO CLIENTE,22,22.0
FUERA DE COBERTURA,16,16.0
SE APERTURARA EL CLIENTE,11,11.0
COMPRA A MAYORISTA,10,10.0
OTROS RUBROS,4,4.0
CLIENTE CADENA,2,2.0


,Cantidad,Porcentaje (%)
Macro resultado,,
DESCARTE,55,55.0
YA ATENDIDO / COBERTURA EXISTENTE,34,34.0
OPORTUNIDAD,11,11.0


tasa de apertura: 11.0%


---
## 4. Rama A — `EXISTE COMO CLIENTE`

El cruce POI ↔ maestra lo propone nuestro motor. Se valida en tres capas:
**nombre**, **dirección** y **coordenadas**.

In [5]:
print("Referencia declarada a la maestra:")
for k, v in m["existe_como_cliente"]["detalle"].items():
    print(f"  {k:38} {v}")

print("\nVeredicto tras validar:")
for k, v in m["veredicto_cruce"].items():
    print(f"  {k:42} {v}")

res["validado"][[f.C_NOMBRE, "pos_name", "score_nombre", "distancia_metros",
                 "veredicto"]]

Referencia declarada a la maestra:
  POS_ID DECLARADO                       16
  CONTRADICCION (dice 'no existe')       4
  TEXTO LIBRE (no verificable)           2

Veredicto tras validar:
  DUPLICADO PROBABLE                         9
  DUPLICADO CONFIRMADO                       3
  POS_ID AUSENTE EN EL CORTE DE LA MAESTRA   2
  REQUIERE REVISION MANUAL                   2


,Código de Cliente interno y Nombre de Establecimiento,pos_name,score_nombre,distancia_metros,veredicto
0,2105.Smoothies Lios,EDVIN OSWALDO TUN CAHUEC,90.57,95.77,DUPLICADO CONFIRMADO
1,1103.Restaurante La Mariscada,DEBORA LITBETH,100.00,576.26,DUPLICADO PROBABLE
2,1424.Restaurante y Piscina El Bucanero,NaN,0.00,NaN,POS_ID AUSENTE EN EL CORTE DE LA MAESTRA
3,1567.Estacion de Servicio Don Rolando,"MARKET DON ROLANDO,",80.85,40748.53,DUPLICADO PROBABLE
4,807.Casa Mía Café,DAVID EDUARDO ALVARADO ESCOBAR,100.00,51.14,DUPLICADO CONFIRMADO
5,138.Familia Ixcot,EMILIO ABIMAEL IXCOT LÓPEZ,100.00,1221.69,DUPLICADO PROBABLE
6,1102.Apoxab,MARTA CHANCHAVAC,91.80,256.59,DUPLICADO PROBABLE
7,810.Gasolinera el Rosario,COMERCIAL SAN MIGUEL,100.00,265.30,DUPLICADO PROBABLE
8,187.Restaurante Tradiciones,NaN,0.00,NaN,POS_ID AUSENTE EN EL CORTE DE LA MAESTRA
9,2647.Casa Campestre,TIENDA EL BUEN APETITO,100.00,141.29,DUPLICADO PROBABLE


### 4.1 Causa raíz — la maestra identifica al titular, no al local

Nuestra base de prospección guarda el **rótulo comercial**; la maestra del
cliente guarda el **titular**. Son dos campos distintos, y el motor comparaba
uno contra el otro.

In [6]:
t = m["titular"]
print(f"el encuestador reescribió el nombre en {t['reescritos']} de {t['evaluados']} casos")
print(f"{t['pct_patron_persona']}% de la maestra ({t['n_patron_persona']:,} registros) "
      f"responde al patrón de nombre de persona".replace(",", "."))
print(f"solo {t['pct_descriptor_comercial']}% lleva un descriptor comercial\n")

for e in t["ejemplos"]:
    print(f"  {e['rotulo']:38} ->  {e['maestra']}")

el encuestador reescribió el nombre en 10 de 14 casos
50.8% de la maestra (116.477 registros) responde al patrón de nombre de persona
solo 28.2% lleva un descriptor comercial

  Smoothies Lios                         ->  EDVIN OSWALDO TUN CAHUEC
  Restaurante La Mariscada               ->  DEBORA LITBETH
  Casa Mia Cafe                          ->  DAVID EDUARDO ALVARADO ESCOBAR
  Familia Ixcot                          ->  EMILIO ABIMAEL IXCOT LÓPEZ
  Apoxab                                 ->  MARTA CHANCHAVAC


### 4.2 Los tres modos de fallo, con sus dos lados

In [7]:
for e in m["ejemplos_h3"]:
    print("=" * 74)
    print(f"{e['modo'].upper()}  ·  {e['glosa']}")
    print(f"  campo    {e['campo_nombre']}")
    print(f"           {e['campo_ctx']}")
    print(f"           lon {e['campo_lon']}   lat {e['campo_lat']}")
    print(f"  maestra  {e['maestra_nombre']}  (pos_id {e['maestra_id']})")
    if e["maestra_lon"] is not None:
        print(f"           {e['maestra_ctx']}")
        print(f"           lon {e['maestra_lon']}   lat {e['maestra_lat']}")
    d = "sin dato" if e["distancia_m"] is None else f"{e['distancia_m']:,} m".replace(",", ".")
    print(f"  score de nombre {e['score']}  ·  distancia {d}")

FALSO NEGATIVO  ·  Ya era cliente y no lo reconocimos: salió a campo como prospecto.
  campo    Kyro's
           GAF · ruta GUP619 · LICORERAS
           lon -89.852695   lat 14.355025
  maestra  KAIROS MARKET  (pos_id 2620040610)
           Asunción Mita · HOGAR CON VENTA
           lon -89.71206   lat 14.33028
  score de nombre 36.6  ·  distancia 15.398 m
CONVENCIÓN DE NOMBRE  ·  El cruce solo funcionó porque el encuestador reescribió el titular.
  campo    Apoxab
           LITORAL · ruta GVP010 · RESTAURANTES
           lon -91.660077   lat 14.571062
  maestra  MARTA CHANCHAVAC  (pos_id 2808012750)
           San Sebastián · INDUSTRIAS
           lon -91.6577   lat 14.570878
  score de nombre 91.8  ·  distancia 257 m
CORTE DESACTUALIZADO  ·  El pos_id es válido en el sistema del embotellador, pero no está en el corte.
  campo    Restaurante y Piscina El Bucanero
           LITORAL · ruta GVP010 · RESTAURANTES
           lon -91.913869   lat 14.291818
  maestra  SIN REGISTRO EN EL 

---
## 5. Rama B — `NO EXISTE EL ESTABLECIMIENTO`

¿El `validity_score` permitía filtrar estos POI antes de mandar a la fuerza de
ventas? Se prueba con separación de distribuciones, posición percentilar y una
curva de decisión.

In [8]:
v = m["validity"]
print(f"AUC = {v['auc']}   (p = {v['p_mw']:.2e})")
print(f"KS  = {v['ks']}   (p = {v['p_ks']:.2e})")
print(f"mediana NO EXISTE {v['mediana_caso']}  vs  resto {v['mediana_control']}")
print(f"casos bajo el percentil 25: {v['casos_bajo_p25']}/{v['n_casos']}")
print(f"controlado por departamento: AUC = {v['auc_controlado']}\n")
print("AUC > 0,50 significa que el score ALTO se asocia a NO EXISTE:")
print("la dirección es la contraria a la esperada.\n")

display(pd.DataFrame(m["curva_umbral"]))
print("lift 1,00 = el umbral no aporta nada sobre elegir al azar")

AUC = 0.724   (p = 4.33e-06)
KS  = 0.476   (p = 8.08e-08)
mediana NO EXISTE 0.949  vs  resto 0.754
casos bajo el percentil 25: 0/35
controlado por departamento: AUC = 0.741

AUC > 0,50 significa que el score ALTO se asocia a NO EXISTE:
la dirección es la contraria a la esperada.



,umbral,recall (% de fallos capturados),% de la base descartada,lift
0,0.30,0.0,0.0,0.00
1,0.40,0.0,18.0,0.00
2,0.50,0.0,29.5,0.00
3,0.60,5.7,36.0,0.16
4,0.70,5.7,47.1,0.12
5,0.75,5.7,49.7,0.11
6,0.80,8.6,55.2,0.16
7,0.85,14.3,58.1,0.25
8,0.90,34.3,64.6,0.53
9,0.95,51.4,74.6,0.69


lift 1,00 = el umbral no aporta nada sobre elegir al azar


---
## 6. Rama C — `FUERA DE COBERTURA`

¿Qué tan lejos está cada punto del cliente activo más cercano?

In [9]:
fc = m["fuera_cobertura"]
print(f"puntos             : {fc['n']}  (con coordenadas: {fc['con_coordenadas']})")
print(f"distancia al vecino: mín {fc['dist_min_m']:.0f} m · "
      f"mediana {fc['dist_mediana_m']:.0f} m · máx {fc['dist_max_m']:.0f} m")
print(f"con un cliente a menos de {fc['radio_m']} m: "
      f"{fc['dentro_radio_n']}/{fc['con_coordenadas']}")

res["ooc"][[f.C_NOMBRE, "Region", "Canal", "dist_cliente_1_m",
            "top_1_pos_name", "top_1_activo"]].sort_values("dist_cliente_1_m")

puntos             : 16  (con coordenadas: 16)
distancia al vecino: mín 32 m · mediana 66 m · máx 199 m
con un cliente a menos de 500 m: 16/16


,Código de Cliente interno y Nombre de Establecimiento,Region,Canal,dist_cliente_1_m,top_1_pos_name,top_1_activo
83,1362.Tienda Doña Maria,GAF,PULPERIA/ABASTECEDOR,31.8,DEPÓSITO LA BENDICIÓN,True
72,2288.TA' RICO,GAF,RESTAURANTES,38.1,VIVIANA YESENIA GUZMAN,False
62,1023.Estacion de Servicio Girasol,GAF,GASOLINERA,40.7,TIENDA MAGNOLIA,True
71,2203.Donde la Vilmita,GAF,RESTAURANTES,41.1,TIENDA MAGNOLIA,True
22,2130.Rancho La Ponderosa,LITORAL,SODA/FONDA,41.2,SARA NOHEMI MERIDA AGUILAR,True
36,816.Panaderia Lorena 2,GAF,HOGAR CON VENTA,42.8,CANCHAS SINTÉTICA CENTROFUT.,True
50,408.Carnicería Darwin,GAF,CARNICERIA,44.4,VENTA REGALITO DE DIOS,True
51,1846.Sabor Caribeno,GAF,RESTAURANTES,64.6,ABARROTERIA FRUTAS Y VERDURAS,True
82,1360.Restaurante El Alboroto,GAF,RESTAURANTES,67.7,MINI TIENDA AZUCENA,True
58,1901.Ristretto Plaza Vieja,GAF,SODA/FONDA,73.0,TIENDA ESQUIPULAS 3,True


---
## 7. Rama D — Mayorista, otros rubros y cadena

Tres etiquetas que **no** son lo mismo: mayorista y otros rubros son riesgo
asumido de prospectar; las cadenas sí eran evitables con nuestro filtro.

In [10]:
display(resumen_categoria(res["rama_d"]["resultado"], "Rama D"))
print(f"cadenas marcadas como tales en nuestra base: "
      f"{m['cadenas_marcadas_en_dataplor']} de {m['rama_d']['CLIENTE CADENA']}")

,Cantidad,Porcentaje (%)
Rama D,,
COMPRA A MAYORISTA,10,62.5
OTROS RUBROS,4,25.0
CLIENTE CADENA,2,12.5


cadenas marcadas como tales en nuestra base: 0 de 2


---
## 8. Dónde ocurrió cada cosa

Las coordenadas se cruzan contra los límites administrativos oficiales
(geoBoundaries gbOpen) para ubicar cada punto y detectar coordenadas fuera del
país.

In [11]:
display(resumen_categoria(res["geo"]["departamento"], "Departamento"))
print(f"puntos fuera del país      : {m['puntos_fuera_del_pais']}")
print(f"sobre la línea de costa    : {m['puntos_en_costa']}  "
      f"(artefacto de generalización, no error del dato)")

display(pd.crosstab(res["geo"]["Region"], res["geo"]["departamento"],
                    margins=True, margins_name="TOTAL"))

,Cantidad,Porcentaje (%)
Departamento,,
Jutiapa,59,59.0
Retalhuleu,24,24.0
Quetzaltenango,8,8.0
Santa Rosa,6,6.0
Guatemala,1,1.0
Jalapa,1,1.0
Escuintla,1,1.0


puntos fuera del país      : 0
sobre la línea de costa    : 5  (artefacto de generalización, no error del dato)


departamento,Escuintla,Guatemala,Jalapa,Jutiapa,Quetzaltenango,Retalhuleu,Santa Rosa,TOTAL
Region,,,,,,,,
ESCUINTLA,1,0,0,0,0,0,0,1
GAF,0,0,1,59,0,0,6,66
GAM,0,1,0,0,0,0,0,1
LITORAL,0,0,0,0,8,24,0,32
TOTAL,1,1,1,59,8,24,6,100


---
## 9. A quién corresponde cada visita

El reparto que sostiene el resumen ejecutivo: **no se solapa y suma el total**.

In [12]:
display(res["cuenta"])
print()
for k, v in m["cuenta_agregada"].items():
    print(f"  {k:22} {v:>3}  ({v / m['visitas_unicas']:.0%})")
print(f"  {'TOTAL':22} {sum(m['cuenta_agregada'].values()):>3}")

,Concepto,Visitas,Responsabilidad,Nota
0,Apertura conseguida,11,RESULTADO ÚTIL,Apertura conseguida: el objetivo de la prueba.
1,Ya era cliente,22,FALLO NUESTRO,Ya era cliente y nuestro filtro no lo excluyó de la lista de prospección.
2,Cadena no detectada,2,FALLO NUESTRO,Cadena no detectada por nuestro filtro: se negocia de forma centralizada.
3,POI que ya no existe,35,DATO DESACTUALIZADO,El POI ya no existe en terreno: la base de prospección está desactualizada.
4,Mal etiquetado «fuera de cobertura»,16,RECUPERABLE,"Mal etiquetado: tiene un cliente de la maestra a menos de 200 m. Es asignación de ruta de preventa, no falta de cobertura."
5,Compra a mayorista,10,RIESGO ASUMIDO,Hay demanda pero la abastece un tercero: solo se descubre visitando.
6,Otros rubros,4,RIESGO ASUMIDO,El punto no pertenece al universo bebible: solo se descubre visitando.



  RESULTADO ÚTIL          11  (11%)
  FALLO NUESTRO           24  (24%)
  DATO DESACTUALIZADO     35  (35%)
  RECUPERABLE             16  (16%)
  RIESGO ASUMIDO          14  (14%)
  TOTAL                  100


---
## 10. Figuras y reporte

Las figuras van a `output/figs/` y el PowerPoint a `output/deck/`. El generador
escribe `..._generado.pptx`: **nunca pisa** la versión que el equipo edita a mano.

In [13]:
figuras.generar(res, PAIS.figs)

  -> 01_resultado_visitas.png

  -> 02_canales.png


  -> 03_validity_score.png
  -> 04_fuera_cobertura.png


  -> 05_mapa_visitas.png


In [14]:
import json

PAIS.metrics.write_text(json.dumps(m, indent=2, ensure_ascii=False), encoding="utf-8")

from src import deck
deck.construir(PAIS)

  -> Reporte_QA_Guatemala_generado.pptx  (26 diapositivas)


WindowsPath('C:/Users/KIN/Desktop/Kin/qa_prospection/data/guatemala/output/deck/Reporte_QA_Guatemala_generado.pptx')

---
## 11. Inventario de lo exportado

Todo archivo de puntos sale con `longitud` y `latitud`: la garantía está en
`Exportador.__call__`, no repartida por el código de análisis.

In [15]:
print(f"{'archivo':38}{'filas':>7}  {'con lon/lat':>12}")
print("-" * 60)
for ruta in sorted(PAIS.findings.glob("*.csv")):
    d = pd.read_csv(ruta, encoding="utf-8-sig", low_memory=False)
    if {"longitud", "latitud"} <= set(d.columns):
        estado = f"{int(d[['longitud', 'latitud']].notna().all(axis=1).sum())}/{len(d)}"
    else:
        estado = "n/a"          # tabla agregada
    print(f"{ruta.name:38}{len(d):>7}  {estado:>12}")

archivo                                 filas   con lon/lat
------------------------------------------------------------
cruces_a_revisar.csv                        2           2/2
cuenta_de_las_visitas.csv                   7           n/a
ejemplos_hallazgo3.csv                      7           7/7
errores_id.csv                              8           8/8
errores_segmentacion.csv                   16         16/16
existe_como_cliente_validado.csv           16         16/16
fuera_de_cobertura.csv                     16         16/16
hallazgos_geo.csv                         100       100/100
hallazgos_resumen.csv                       8           n/a
poi_inexistentes.csv                       35         35/35
pos_id_ausente_en_corte.csv                 2           2/2
titular_vs_nombre_comercial.csv            14         14/14
